# RetailIQ — Big Data Analytics & Retail Decision Intelligence
### Portfolio Project: Exploratory Data Analysis, SQL Intelligence, RFM Segmentation & Forecasting
**Author:** Computer Science & Engineering Undergraduate (Specialization in Big Data Analytics)

---

## 1. Project Background & Business Objectives
Modern retail enterprises face severe challenges in balancing inventory capital, identifying churn risk across recurring customer cohorts, and uncovering low-margin or loss-leader SKUs. 

**Key Business Questions Answered:**
1. **Revenue Drivers:** Which product categories and geographic regions generate the greatest top-line and bottom-line margin?
2. **Order Integrity:** What is the true Average Order Value (AOV) when accounting for multi-item orders without double-counting?
3. **Customer Value:** How is spend distributed across customer tiers via Recency, Frequency, and Monetary (RFM) quintile analysis?
4. **Churn Prevention:** Which high-value accounts have been inactive for >60 days?
5. **Demand Horizon:** What is the projected sales trajectory over the upcoming periods using Holt's Linear Trend smoothing?

## 2. Ingestion & Environment Setup
We load the transactional dataset, establish an in-memory SQLite database, and verify column schemas.

In [ ]:
import csv
import sqlite3
import math
from datetime import datetime
from collections import defaultdict

# Load CSV into structured memory
csv_path = '../backend/data/sample_retail_transactions.csv'
transactions = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append({
            'id': row['id'],
            'order_id': row['order_id'],
            'order_date': row['order_date'],
            'customer_id': row['customer_id'],
            'product_id': row['product_id'],
            'product_name': row['product_name'],
            'category': row['category'],
            'quantity': float(row['quantity']),
            'unit_price': float(row['unit_price']),
            'sales': float(row['sales']),
            'cost': float(row['cost']),
            'profit': float(row['profit']),
            'region': row['region'],
            'state': row['state'],
            'payment_method': row['payment_method'],
            'discount': float(row['discount'])
        })

print(f'Ingested {len(transactions):,} retail transaction records from {csv_path}.')

## 3. Data Quality & Integrity Audit
Rigorous business intelligence requires validating data quality before executing business calculations.
- Checking for missing cost coverage (to ensure profit isn't fabricated).
- Identifying negative quantities representing customer returns/credit notes.
- Checking for duplicate transaction IDs.

In [ ]:
total_rows = len(transactions)
unique_tx_ids = set(t['id'] for t in transactions)
unique_orders = set(t['order_id'] for t in transactions)
unique_customers = set(t['customer_id'] for t in transactions)

missing_cost_count = sum(1 for t in transactions if t['cost'] is None or math.isnan(t['cost']))
return_rows = [t for t in transactions if t['quantity'] < 0]
cost_coverage_pct = ((total_rows - missing_cost_count) / total_rows) * 100

print(f'=== DATA QUALITY HEALTH AUDIT ===')
print(f'Total Transaction Rows:       {total_rows:,}')
print(f'Distinct Order Groups:        {len(unique_orders):,} (avg {total_rows/len(unique_orders):.2f} items/order)')
print(f'Unique Active Customers:      {len(unique_customers):,}')
print(f'Duplicate Transaction IDs:    {total_rows - len(unique_tx_ids)}')
print(f'Cost Coverage Rate:           {cost_coverage_pct:.1f}% (Health Threshold: >= 50%)')
print(f'Customer Return Transactions: {len(return_rows)} records (Negative units)')

## 4. Enterprise KPI Calculations
We enforce strict metric definitions:
- **Total Revenue**: $\sum \text{sales}$
- **Total Profit**: $\sum \text{sales} - \sum \text{COGS}$
- **Profit Margin %**: $(\text{Total Profit} / \text{Total Revenue}) \times 100$
- **Average Order Value (AOV)**: $\text{Total Revenue} / \text{Count of Distinct Orders}$
- **Average Selling Price (ASP)**: $\text{Net Revenue} / \text{Net Units Sold}$
- **Return Rate %**: $(\text{Return Units} / (\text{Sold Units} + \text{Return Units})) \times 100$

In [ ]:
total_revenue = sum(t['sales'] for t in transactions)
total_cost = sum(t['cost'] for t in transactions)
total_profit = sum(t['profit'] for t in transactions)
profit_margin = (total_profit / total_revenue) * 100 if total_revenue > 0 else 0.0

num_distinct_orders = len(unique_orders)
aov = total_revenue / num_distinct_orders if num_distinct_orders > 0 else 0.0

num_distinct_customers = len(unique_customers)
arpc = total_revenue / num_distinct_customers if num_distinct_customers > 0 else 0.0

positive_units = sum(t['quantity'] for t in transactions if t['quantity'] > 0)
return_units = abs(sum(t['quantity'] for t in transactions if t['quantity'] < 0))
net_units_sold = positive_units - return_units
asp = total_revenue / net_units_sold if net_units_sold > 0 else 0.0
return_rate_pct = (return_units / positive_units) * 100 if positive_units > 0 else 0.0

print(f'=== EXECUTIVE FINANCIAL & OPERATIONAL KPIS ===')
print(f'Total Net Sales (Revenue):    ${total_revenue:,.2f}')
print(f'Total Cost of Goods (COGS):   ${total_cost:,.2f}')
print(f'Net Enterprise Profit:        ${total_profit:,.2f}')
print(f'Operating Profit Margin:      {profit_margin:.2f}%')
print(f'Average Order Value (AOV):    ${aov:.2f}')
print(f'Average Revenue Per Customer: ${arpc:.2f}')
print(f'Average Selling Price (ASP):  ${asp:.2f}')
print(f'Customer Return Rate:         {return_rate_pct:.2f}% ({return_units:,.0f} units returned)')

## 5. Exploratory Data Analysis & Statistical Distribution
We inspect parametric (Mean, StDev) and non-parametric (Median, IQR Fences) statistics to investigate skewness and outliers.

In [ ]:
sales_vals = sorted([t['sales'] for t in transactions if t['sales'] > 0])
n = len(sales_vals)
mean_val = sum(sales_vals) / n
median_val = sales_vals[n // 2] if n % 2 != 0 else (sales_vals[n//2 - 1] + sales_vals[n//2]) / 2
variance = sum((x - mean_val)**2 for x in sales_vals) / n
std_dev = math.sqrt(variance)

# Tukey's 1.5x IQR Fences
q1 = sales_vals[int(n * 0.25)]
q3 = sales_vals[int(n * 0.75)]
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
outliers = [x for x in sales_vals if x > upper_fence]

print(f'=== SALES DISTRIBUTION & STATISTICAL METRICS ===')
print(f'Mean Transaction Value:     ${mean_val:.2f}')
print(f'Median Transaction Value:   ${median_val:.2f}')
print(f'Standard Deviation:         ${std_dev:.2f}')
print(f'Q1 (25th Percentile):       ${q1:.2f}')
print(f'Q3 (75th Percentile):       ${q3:.2f}')
print(f'Interquartile Range (IQR):  ${iqr:.2f}')
print(f'Upper Outlier Fence:        ${upper_fence:.2f}')
print(f'Statistical Outliers Found: {len(outliers)} orders ({(len(outliers)/n)*100:.2f}% of volume)')

## 6. Category & Regional Contribution Analysis
We evaluate which product divisions and geographic markets generate the strongest margins.

In [ ]:
category_metrics = defaultdict(lambda: {'sales': 0.0, 'profit': 0.0, 'units': 0})
for t in transactions:
    c = t['category']
    category_metrics[c]['sales'] += t['sales']
    category_metrics[c]['profit'] += t['profit']
    category_metrics[c]['units'] += t['quantity']

print(f'{"Category":<18} | {"Net Sales":<14} | {"Net Profit":<14} | {"Margin %":<10} | {"Units Sold":<10}')
print('-' * 76)
for cat, data in sorted(category_metrics.items(), key=lambda x: x[1]['sales'], reverse=True):
    margin = (data['profit'] / data['sales'] * 100) if data['sales'] > 0 else 0.0
    print(f'{cat:<18} | ${data["sales"]:>12,.2f} | ${data["profit"]:>12,.2f} | {margin:>8.2f}% | {data["units"]:>10,.0f}')

## 7. Customer RFM Segmentation Analysis
We segment customers along 3 behavioral dimensions:
1. **Recency**: Days elapsed since customer's latest order.
2. **Frequency**: Count of distinct completed orders.
3. **Monetary**: Cumulative net spend across all orders.

In [ ]:
customer_orders = defaultdict(set)
customer_spend = defaultdict(float)
customer_last_date = defaultdict(str)

for t in transactions:
    cid = t['customer_id']
    customer_orders[cid].add(t['order_id'])
    customer_spend[cid] += t['sales']
    if t['order_date'] > customer_last_date[cid]:
        customer_last_date[cid] = t['order_date']

max_date = datetime.strptime(max(t['order_date'] for t in transactions), '%Y-%m-%d')

rfm_table = []
for cid in customer_orders:
    last_dt = datetime.strptime(customer_last_date[cid], '%Y-%m-%d')
    recency_days = (max_date - last_dt).days
    frequency = len(customer_orders[cid])
    monetary = customer_spend[cid]
    rfm_table.append({'customer_id': cid, 'recency': recency_days, 'frequency': frequency, 'monetary': monetary})

# Segment classification
def classify(r, f, m):
    if r <= 45 and f >= 4 and m >= 1200:
        return 'High-Value VIPs'
    elif f >= 3:
        return 'Loyal Repeat Customers'
    elif r <= 60 and f >= 2:
        return 'Potential Loyalists'
    elif r <= 30:
        return 'Recent New Buyers'
    elif r > 120:
        return 'At-Risk / Lapsed'
    else:
        return 'Needing Attention'

segment_counts = defaultdict(lambda: {'count': 0, 'revenue': 0.0})
for cust in rfm_table:
    seg = classify(cust['recency'], cust['frequency'], cust['monetary'])
    segment_counts[seg]['count'] += 1
    segment_counts[seg]['revenue'] += cust['monetary']

print(f'{"RFM Segment":<24} | {"Customer Count":<16} | {"Total Revenue":<14} | {"Rev Share %":<12}')
print('-' * 74)
for seg, data in sorted(segment_counts.items(), key=lambda x: x[1]['revenue'], reverse=True):
    share = (data['revenue'] / total_revenue) * 100
    print(f'{seg:<24} | {data["count"]:>16,} | ${data["revenue"]:>12,.2f} | {share:>10.2f}%')

## 8. Analytical SQL Studio (Window Functions & Inactivity CTEs)
We initialize an in-memory SQLite relational instance to execute enterprise SQL queries.

In [ ]:
conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# Create normalized schema
cur.execute('''
CREATE TABLE transactions (
    id TEXT PRIMARY KEY,
    order_id TEXT NOT NULL,
    order_date TEXT NOT NULL,
    customer_id TEXT,
    product_id TEXT,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity REAL NOT NULL,
    unit_price REAL NOT NULL,
    sales REAL NOT NULL,
    cost REAL NOT NULL,
    profit REAL NOT NULL,
    region TEXT,
    state TEXT,
    payment_method TEXT,
    discount REAL
);
''')

for t in transactions:
    cur.execute('''
    INSERT INTO transactions VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (t['id'], t['order_id'], t['order_date'], t['customer_id'], t['product_id'],
          t['product_name'], t['category'], t['quantity'], t['unit_price'], t['sales'],
          t['cost'], t['profit'], t['region'], t['state'], t['payment_method'], t['discount']))
conn.commit()

# Query: Category Product Rankings via SQL Window Function DENSE_RANK()
cur.execute('''
WITH CategoryTotals AS (
    SELECT category, product_name, ROUND(SUM(sales), 2) as item_sales, SUM(quantity) as units
    FROM transactions
    GROUP BY category, product_name
)
SELECT category, product_name, item_sales, units,
       DENSE_RANK() OVER (PARTITION BY category ORDER BY item_sales DESC) as rank_in_category
FROM CategoryTotals
ORDER BY category, rank_in_category ASC;
''')
top_ranked = cur.fetchall()[:10]

print('Top Ranked Products by Category (SQL Window Function DENSE_RANK):')
for row in top_ranked:
    print(f'  [{row[0]}] Rank {row[4]}: {row[1]} (${row[2]:,.2f}, {row[3]:.0f} units)')

## 9. Chronological Time-Series Sales Forecasting (Holt's Linear Smoothing)
We aggregate transactions into monthly series, split into chronological train/test sets, and evaluate Mean Absolute Error (MAE) against a Naive Baseline.

In [ ]:
monthly_sales = defaultdict(float)
for t in transactions:
    month_key = t['order_date'][:7]
    monthly_sales[month_key] += t['sales']

sorted_months = sorted(monthly_sales.keys())
series = [monthly_sales[m] for m in sorted_months]

# Train / Test chronological split (75% train, 25% test)
split = int(len(series) * 0.75)
train, test = series[:split], series[split:]

# Fit Holt's Linear Trend
alpha, beta = 0.35, 0.15
level = train[0]
trend = (train[-1] - train[0]) / max(1, len(train) - 1)

for val in train[1:]:
    prev_level = level
    level = alpha * val + (1 - alpha) * (prev_level + trend)
    trend = beta * (level - prev_level) + (1 - beta) * trend

# Generate test predictions
preds = [max(0, level + (h + 1) * trend) for h in range(len(test))]
holt_mae = sum(abs(a - p) for a, p in zip(test, preds)) / len(test) if test else 0.0

# Naive baseline (last observed value)
naive_preds = [train[-1]] * len(test)
naive_mae = sum(abs(a - p) for a, p in zip(test, naive_preds)) / len(test) if test else 0.0

print(f'=== TIME-SERIES MODEL EVALUATION ===')
print(f'Observations Analyzed:  {len(series)} months ({sorted_months[0]} to {sorted_months[-1]})')
print(f'Training Window:        {len(train)} months | Out-of-Sample Test: {len(test)} months')
print(f'Holt\'s Linear MAE:      ${holt_mae:,.2f}')
print(f'Naive Baseline MAE:     ${naive_mae:,.2f}')
mae_improvement = ((naive_mae - holt_mae) / naive_mae) * 100 if naive_mae > 0 else 0.0
print(f'Holt\'s Model Accuracy vs Baseline: {mae_improvement:+.1f}% improvement')

# Forecast Horizon (Next 3 Months)
next_3m_forecast = [level + (len(test) + h) * trend for h in range(1, 4)]
print(f'3-Month Horizon Forecast: {[f"${f:,.2f}" for f in next_3m_forecast]}')

## 10. Analytical Synthesis & Recommendations

### Evidence-Based Findings:
1. **Margin Discrepancies**: High revenue contribution does not always guarantee healthy margins. Categories like Electronics drive heavy volume but require pricing optimization on accessories (e.g. Clearance USB Cable shows negative margins).
2. **Customer Retention**: High-value VIPs and Repeat loyalists contribute a disproportionate share of enterprise revenue. Retention campaigns targeting accounts inactive for >60 days are vital to prevent churn.
3. **Return Rate Impact**: Returns represent a quantifiable friction point in net units and logistics cost. Monitoring SKU return rate helps spot manufacturing defects early.
4. **Forecasting**: Holt's Linear Trend smoothing provides reliable directional signals with measurable improvement over naive random walk baselines.

---